In [1]:
import numpy as np
from voxeloo import culling
from PIL import Image

In [2]:
def perspective_matrix(aspect, fov, near, far):
    """Returns the perspective projection matrix in row-major order."""
    n = near
    f = far
    t = np.tan(0.5 * fov) * near
    b = -t
    r = t * aspect
    l = -r

    # make the projection matrix
    return np.array([
        [2 * n / (r - l), 0, (r + l) / (r - l), 0],
        [0, 2 * n / (t - b), (t + b) / (t - b), 0],
        [0, 0, -(f + n) / (f - n), -2 * f * n / (f - n)],
        [0, 0, -1, 0],
    ])

In [3]:
def random_aabb():
    pos = 2.0 * np.random.rand(3) - 1.0
    pos[0] *= 50
    pos[1] *= 50
    pos[2] = 100 * pos[2] - 150
    return culling.AABB(
        v0=pos,
        v1=pos + [4, 4, -10],
    )

In [4]:
proj = perspective_matrix(
    aspect=1,
    fov=0.5 * np.pi,
    near=0.1,
    far=256.0,
)

In [32]:
proj.dot([0, 0, 0.001, 1])

array([ 0.        ,  0.        , -0.20107894, -0.001     ])

In [5]:
aabbs = [random_aabb() for _ in range(1000)]

In [6]:
BUFFER_SHAPE = (256, 256)

### Do a overestimating conservative rasterization

In [ ]:
buffer = culling.OcclusionBuffer(*BUFFER_SHAPE)

In [ ]:
%%time

culling.rasterize_many_aabb_inclusive(
    buffer,
    proj.transpose().flatten(),  # Column-major
    aabbs,
)

In [ ]:
inclusive = np.array([
    buffer.get(x, y)
    for y in range(BUFFER_SHAPE[1])
    for x in range(BUFFER_SHAPE[0])
])
pixels = 255 * inclusive.reshape(*BUFFER_SHAPE).astype(np.uint8)

In [ ]:
Image.fromarray(pixels)

### Do a underestimating conservative rasterization

In [ ]:
buffer = culling.OcclusionBuffer(*BUFFER_SHAPE)

In [ ]:
%%time

culling.rasterize_many_aabb_exclusive(
    buffer,
    proj.transpose().flatten(),  # Column-major
    aabbs,
)

In [ ]:
exclusive = np.array([
    buffer.get(x, y)
    for y in range(BUFFER_SHAPE[1])
    for x in range(BUFFER_SHAPE[0])
])
pixels = 255 * exclusive.reshape(*BUFFER_SHAPE).astype(np.uint8)

In [ ]:
Image.fromarray(pixels)

### Render the difference

In [ ]:
difference = inclusive & ~exclusive
pixels = 255 * difference.reshape(*BUFFER_SHAPE).astype(np.uint8)

In [ ]:
Image.fromarray(pixels)

### Try out the occlusion culler

In [ ]:
culler = culling.OcclusionCuller(
    proj=proj.transpose().flatten(),  # Column-major
    shape=BUFFER_SHAPE,
)

In [ ]:
%%time 

for aabb in aabbs:
    culler.write(aabb)

In [ ]:
%%time 

buffer = culling.OcclusionBuffer(*BUFFER_SHAPE)
for aabb in [random_aabb() for _ in range(1000)]:
    if not culler.test(aabb):
        culling.rasterize_aabb_inclusive(
            buffer,
            proj.transpose().flatten(),  # Column-major
            aabb,
        )

In [ ]:
passes = np.array([
    buffer.get(x, y)
    for y in range(BUFFER_SHAPE[1])
    for x in range(BUFFER_SHAPE[0])
])

pixels = np.zeros(shape=(*BUFFER_SHAPE, 3), dtype=np.uint8)
pixels[exclusive.reshape(*BUFFER_SHAPE), :] = 255
pixels[passes.reshape(*BUFFER_SHAPE), :] = [255, 0, 0]

In [ ]:
Image.fromarray(pixels)